# Agri Vision — three-client GPU training

Select **Runtime → Change runtime type → GPU**, edit the settings below if needed, then **Runtime → Run all** and approve Drive access.

Upload the updated `original_dataset_colab.zip`, `agrivision_colab_code.zip`, and `saved_models/global_round_020.pt` from your project folder to `MyDrive/Agri Vision/` first. Replace the old dataset/code ZIPs in Drive. The notebook is configured to continue rounds 21-25 from the verified round-20 checkpoint at image size 640. The updated dataset has 15,512 training, 4,374 validation, and 2,191 held-out test images. The code bundle supplies the corrected 30-class YAML. It runs three clients sequentially on one GPU and performs sample-weighted FedAvg after all three finish. This is a local simulation; no distributed workers or exposed server are started.

GPU availability and session duration are limited. Only completed global rounds can resume; interruption mid-round requires repeating that round. The class names currently in GitHub were reconstructed; passing numeric validation does not independently verify annotation semantics. Verify names against your original dataset export before reporting disease-specific accuracy.

In [ ]:
from pathlib import Path
ROUNDS = 5
LOCAL_EPOCHS = 1
BATCH = 8
IMGSZ = 640
WORKERS = 2
EXPECTED_START_ROUND = 20
EXPECTED_SPLIT_COUNTS = {'train': 15512, 'valid': 4374, 'test': 2191}
EXPECTED_DATASET_SHA256 = 'a21b497dbced6a8797e3f10fa4af6baafad734533417b37fff6f149deabc0827'
EXPECTED_CODE_SHA256 = 'dc625e362543d0c25a34f9b8ab2ad08b1b35b9b133152776ea60255528328896'
EXPECTED_CHECKPOINT_SHA256 = '0d22b00f1143ce522d7793c9029a1cce381e75d3a568116648107b35544c5657'
DATASET_ZIP = '/content/drive/MyDrive/Agri Vision/original_dataset_colab.zip'
CODE_ZIP = '/content/drive/MyDrive/Agri Vision/agrivision_colab_code.zip'
OUTPUT_ROOT = '/content/drive/MyDrive/Agri Vision/colab_results'
# Required starting point for this rounds 21-25 continuation
INITIAL_CHECKPOINT = '/content/drive/MyDrive/Agri Vision/global_round_020.pt'


In [ ]:
from google.colab import drive
import hashlib
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then Run all again.')
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
if not Path(DATASET_ZIP).is_file():
    raise FileNotFoundError(f'Upload original_dataset_colab.zip here first: {DATASET_ZIP}')
if not Path(CODE_ZIP).is_file():
    raise FileNotFoundError(f'Upload agrivision_colab_code.zip here first: {CODE_ZIP}')
if INITIAL_CHECKPOINT and not Path(INITIAL_CHECKPOINT).is_file():
    raise FileNotFoundError(f'Upload the round-20 checkpoint here first: {INITIAL_CHECKPOINT}')
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
verified_hashes = {
    'dataset': sha256_file(DATASET_ZIP),
    'code': sha256_file(CODE_ZIP),
    'checkpoint': sha256_file(INITIAL_CHECKPOINT),
}
expected_hashes = {
    'dataset': EXPECTED_DATASET_SHA256,
    'code': EXPECTED_CODE_SHA256,
    'checkpoint': EXPECTED_CHECKPOINT_SHA256,
}
if verified_hashes != expected_hashes:
    raise ValueError(f'An uploaded file is old or incomplete. Expected {expected_hashes}, found {verified_hashes}')
print('Verified all three upload checksums.')
checkpoint_metadata = torch.load(INITIAL_CHECKPOINT, map_location='cpu', weights_only=True)
if int(checkpoint_metadata.get('round', -1)) != EXPECTED_START_ROUND:
    raise ValueError(f'Expected a round-{EXPECTED_START_ROUND} checkpoint, found round {checkpoint_metadata.get("round")}')
print('Verified starting checkpoint round:', checkpoint_metadata['round'])
del checkpoint_metadata


## Load the corrected project code

The small Drive bundle avoids GitHub authentication and excludes the Windows virtual environment. Its SHA-256 hash is recorded so the experiment can be reproduced.

In [ ]:
import json, uuid, os, subprocess, sys, zipfile
WORK = Path('/content') / ('agrivision_' + uuid.uuid4().hex[:8])
WORK.mkdir()
CODE_BUNDLE_SHA256 = verified_hashes['code']
with zipfile.ZipFile(CODE_ZIP) as bundle:
    bundle.extractall(WORK)
os.chdir(WORK)
print('Code bundle SHA-256:', CODE_BUNDLE_SHA256)
print('Working directory:', WORK)
# Keep Colab GPU PyTorch; do not copy the Windows environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK / 'requirements.txt')], check=True, cwd=WORK)
# Verify CUDA again in a fresh process after dependency installation.
subprocess.run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(), 'CUDA unavailable after install'; print('Training GPU:', torch.cuda.get_device_name(0))"], check=True, cwd=WORK)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)


## Prepare data on Colab's local disk

The ZIP must contain `original_dataset/train`, `valid`, and `test` with `images` and `labels`. The notebook extracts only images and label text, preserves the corrected YAML from the code bundle, and rebuilds the three disjoint shards with seed 42. The rebuild runs directly in Python so any error is displayed without a subprocess wrapper.

In [ ]:
import shutil, zipfile
archive = WORK / 'source.zip'
shutil.copy2(DATASET_ZIP, archive)
with zipfile.ZipFile(archive) as z:
    for entry in z.infolist():
        parts = Path(entry.filename).parts
        if entry.is_dir():
            continue
        if len(parts) != 4 or parts[0] != 'original_dataset' or parts[1] not in ('train', 'valid', 'test') or parts[2] not in ('images', 'labels'):
            continue
        suffix = Path(parts[3]).suffix.lower()
        if (parts[2] == 'labels' and suffix != '.txt') or (parts[2] == 'images' and suffix not in ('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
            continue
        target = WORK.joinpath(*parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(entry) as source, target.open('wb') as dest:
            shutil.copyfileobj(source, dest)
actual_counts = {split: len(list((WORK / 'original_dataset' / split / 'images').iterdir())) for split in EXPECTED_SPLIT_COUNTS}
if actual_counts != EXPECTED_SPLIT_COUNTS:
    raise ValueError(f'Wrong or incomplete dataset ZIP: expected {EXPECTED_SPLIT_COUNTS}, found {actual_counts}')
print('Verified updated dataset counts:', actual_counts)
sys.path.insert(0, str(WORK))
from split_dataset import rebuild
backup, report = rebuild(seed=42)
for line in report:
    print(line)
print('Original YAML-only dataset template preserved at:', backup)


## Train and save to Drive

Each round trains client1, client2, then client3 from the same starting global model. A global checkpoint is saved immediately after aggregation, followed by validation metrics. Live output is shown below and copied to Drive. Stop the cell to stop its training process. Select a new run to restart; never overwrite an earlier run.

In [ ]:
from datetime import datetime, timezone
RUN = Path(OUTPUT_ROOT) / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
RUN.parent.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', str(WORK / 'colab_train.py'), '--rounds', str(ROUNDS), '--local-epochs', str(LOCAL_EPOCHS), '--batch', str(BATCH), '--imgsz', str(IMGSZ), '--workers', str(WORKERS), '--output', str(RUN)]
if INITIAL_CHECKPOINT:
    if not Path(INITIAL_CHECKPOINT).is_file():
        raise FileNotFoundError(INITIAL_CHECKPOINT)
    command += ['--initial-checkpoint', INITIAL_CHECKPOINT]
log_path = RUN.parent / (RUN.name + '.log')
(RUN.parent / (RUN.name + '_input_sha256.json')).write_text(json.dumps(verified_hashes, indent=2))
print('Results:', RUN)
print('Log:', log_path)
with log_path.open('w') as log:
    process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
        if result:
            raise RuntimeError(f'Training exited with code {result}; inspect the log above.')
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
print('Saved checkpoints:', list(RUN.glob('global_round_*.pt')))


## Next session

Use the newest completed `global_round_XXX.pt` as `INITIAL_CHECKPOINT`, update `EXPECTED_START_ROUND` and `EXPECTED_CHECKPOINT_SHA256`, choose how many **additional** rounds to train, then Run all. The dataset remains on Drive; local copies disappear when the runtime is recycled. Old 23-class checkpoints are incompatible. Validation metrics are reported each round; keep the test split for the final evaluation. One successful run confirms execution, not model accuracy.